# Fase 0 — Exploración y limpieza de fuentes · Alcaldía Local de San Cristóbal

Este notebook es el **laboratorio de datos** del proyecto (ver `docs/PRD.md`). Aquí se descargan, exploran y limpian las fuentes oficiales que alimentarán al agente de IA.

**Cómo usarlo (no necesitas saber programar):**
1. Menú **Entorno de ejecución → Ejecutar todas** (o `Ctrl + F9`).
2. Lee los textos entre las celdas de código: explican qué se hace y por qué.
3. Al final, en **Hallazgos y tareas para el equipo**, escribe tus observaciones en la celda correspondiente.
4. Para descargar los datos limpios, abre el panel de archivos (ícono de carpeta, a la izquierda) → `data/processed`.

| Fuente | Qué responde | Origen |
|---|---|---|
| **F1** Ejecución PAC — Fondos de Desarrollo Local | ¿Cuánto dinero programado se giró realmente cada mes? | Secretaría Distrital de Hacienda |
| **F2** PQRS Secretaría de Gobierno 2022–2026 | ¿Qué pide y de qué se queja la ciudadanía? ¿Qué tan rápido se responde? | Secretaría Distrital de Gobierno |
| **F3** Presupuestos Participativos | ¿Qué priorizó la comunidad y cómo avanza? | Secretaría Distrital de Planeación |

> **Regla del proyecto:** este notebook es el *laboratorio*. Cuando una limpieza quede definida aquí, se traslada al repositorio (`backend/ingestion/`) con su prueba. La aplicación nunca lee datos directamente de este notebook.

## 1. Preparación del entorno

In [ ]:
%pip install -q odfpy

import csv
import io
import json
import re
import unicodedata
from datetime import datetime, timezone
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import requests

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)

CKAN_API = "https://datosabiertos.bogota.gov.co/api/3/action"
LOCALITY_CODE = 4  # San Cristóbal
LOCALITY_NAME = "San Cristóbal"

RAW_DIR = Path("data/raw")
PROCESSED_DIR = Path("data/processed")
RAW_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

# Traceability record: every processed file documents where it came from.
SOURCES_METADATA = []

### Funciones auxiliares
- `get_dataset`: consulta la API del portal (CKAN) y devuelve la descripción del dataset con sus archivos.
- `download`: descarga un archivo una sola vez (si ya existe, lo reutiliza).
- `normalize_column`: convierte nombres como `% de avance  propuesta` en `pct_de_avance_propuesta`, más fáciles de usar.

In [ ]:
def get_dataset(dataset_id: str) -> dict:
    response = requests.get(f"{CKAN_API}/package_show", params={"id": dataset_id}, timeout=60)
    response.raise_for_status()
    return response.json()["result"]


def download(url: str, filename: str) -> Path:
    path = RAW_DIR / filename
    if not path.exists():
        response = requests.get(url, timeout=300)
        response.raise_for_status()
        path.write_bytes(response.content)
    return path


def strip_accents(text: str) -> str:
    return "".join(c for c in unicodedata.normalize("NFKD", str(text)) if not unicodedata.combining(c))


def normalize_column(name: str) -> str:
    name = strip_accents(name).lower().replace("%", "pct")
    return re.sub(r"[^a-z0-9]+", "_", name).strip("_")


def register_source(source_id, title, dataset_id, resource_urls, output_file, cutoff, rows):
    SOURCES_METADATA.append({
        "source_id": source_id,
        "title": title,
        "portal_url": f"https://datosabiertos.bogota.gov.co/dataset/{dataset_id}",
        "resource_urls": resource_urls,
        "output_file": str(output_file),
        "data_cutoff": cutoff,
        "rows": int(rows),
        "ingested_at": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    })

## 2. F1 — Ejecución PAC de los Fondos de Desarrollo Local

El **PAC** (Programa Anual Mensualizado de Caja) es el dinero que cada Fondo de Desarrollo Local programa para girar cada mes. El dataset publica un archivo CSV **por mes**, con las 20 localidades.

- `PAC ACTUAL`: recursos programados para el mes.
- `EJECUTADO`: recursos efectivamente girados (salida de bancos).
- `% EJECUTADO`: ejecutado / programado.

**Problemas de calidad detectados al inspeccionar los archivos:**
- No son tablas limpias: traen ~10 filas de encabezado decorativo antes de los datos.
- Codificación `latin-1` (las tildes se ven mal si se abre como UTF-8).
- Números con punto de miles (`7.980`) y porcentajes como texto (`99%`).
- **El archivo no indica la unidad** de los valores. Por magnitud parecen **millones de pesos**, pero hay que confirmarlo (ver tareas al final).

In [ ]:
PAC_DATASET_ID = "e130bb0d-2d92-41c8-b894-8e058519309d"
MONTHS = {"ENERO": 1, "FEBRERO": 2, "MARZO": 3, "ABRIL": 4, "MAYO": 5, "JUNIO": 6, "JULIO": 7,
          "AGOSTO": 8, "SEPTIEMBRE": 9, "OCTUBRE": 10, "NOVIEMBRE": 11, "DICIEMBRE": 12}

pac_dataset = get_dataset(PAC_DATASET_ID)
pac_resources = [r for r in pac_dataset["resources"]
                 if r["format"].upper() == "CSV" and "metadata" not in r["name"].lower()]
print(f"{pac_dataset['title']}: {len(pac_resources)} archivos mensuales")

In [ ]:
def decode(raw: bytes) -> str:
    try:
        return raw.decode("utf-8-sig")
    except UnicodeDecodeError:
        return raw.decode("latin-1")


def to_number(value: str) -> float | None:
    value = (value or "").strip().replace("%", "").replace(".", "").replace(",", ".")
    try:
        return float(value)
    except ValueError:
        return None


def find_period(rows: list[list[str]], fallback_name: str) -> tuple[int, int]:
    # Month and year live in a header row like ';MES:;JUNIO;;AÑO;2025'. Fallback: the resource name.
    for row in rows[:12]:
        joined = strip_accents(" ".join(row)).upper()
        if "MES" in joined:
            month = next((m for name, m in MONTHS.items() if name in joined), None)
            year = re.search(r"20\d{2}", joined)
            if month and year:
                return int(year.group()), month
    name = strip_accents(fallback_name).upper()
    month = next(m for n, m in MONTHS.items() if n in name)
    return int(re.search(r"20\d{2}", name).group()), month


def parse_pac_file(raw: bytes, resource_name: str) -> pd.DataFrame:
    text = decode(raw)
    delimiter = ";" if text.count(";") > text.count(",") else ","
    rows = list(csv.reader(io.StringIO(text), delimiter=delimiter))
    year, month = find_period(rows, resource_name)
    header_idx = next(i for i, row in enumerate(rows)
                      if any(strip_accents(c).strip().upper() == "CODIGO" for c in row))
    header = [strip_accents(c).strip().upper() for c in rows[header_idx]]
    col = {name: header.index(name) for name in ("CODIGO", "ENTIDAD")}
    records = []
    for row in rows[header_idx + 1:]:
        if len(row) <= col["ENTIDAD"] + 4 or not re.fullmatch(r"\d{4}-\d{2}", row[col["CODIGO"]].strip()):
            continue  # keep only entity rows like '0004-01'; skips blanks, 'Total' and footer tables
        base = col["ENTIDAD"]
        records.append({
            "year": year,
            "month": month,
            "entity_code": row[col["CODIGO"]].strip(),
            "locality": strip_accents(row[base]).strip(),
            "pac_programmed": to_number(row[base + 1]),
            "pac_executed": to_number(row[base + 2]),
            "pct_executed_reported": (to_number(row[base + 3]) or 0) / 100,
        })
    return pd.DataFrame(records)


frames, failures = [], []
for resource in pac_resources:
    try:
        raw = download(resource["url"], f"pac_{resource['id']}.csv").read_bytes()
        frames.append(parse_pac_file(raw, resource["name"]))
    except Exception as error:  # a failing file is itself a data-quality finding
        failures.append((resource["name"], repr(error)))

pac_all = pd.concat(frames, ignore_index=True).sort_values(["year", "month", "locality"])
print(f"Filas: {len(pac_all)} | archivos con error: {len(failures)}")
for name, error in failures:
    print("  ERROR:", name, error)

# Every month must contain exactly the 20 localities.
rows_per_month = pac_all.groupby(["year", "month"]).size()
print("Meses con un número de localidades distinto de 20:", rows_per_month[rows_per_month != 20].to_dict())
pac_all.head()

### Validación
Antes de confiar en los datos se verifica que el porcentaje **publicado** coincida con el porcentaje **calculado** (`ejecutado / programado`). Si no coinciden, hay un error en la fuente o en nuestra limpieza.

In [ ]:
pac_all["pct_executed_calc"] = (pac_all["pac_executed"] / pac_all["pac_programmed"]).round(2)
mismatch = pac_all[(pac_all["pct_executed_calc"] - pac_all["pct_executed_reported"]).abs() > 0.015]
missing = pac_all[pac_all[["pac_programmed", "pac_executed"]].isna().any(axis=1)]
print(f"Filas donde el % publicado no coincide con el calculado: {len(mismatch)} de {len(pac_all)}")
print(f"Filas con un valor vacío en la fuente: {len(missing)}")
missing

In [ ]:
# Locality ranking per month (1 = best execution) — useful context: how does San Cristóbal compare?
pac_all["rank_in_month"] = (pac_all.groupby(["year", "month"])["pct_executed_calc"]
                            .rank(ascending=False, method="min").astype("Int64"))  # NA when the source value is blank

pac_sc = pac_all[pac_all["entity_code"].str.startswith("0004")].copy()
pac_sc["period"] = pd.to_datetime(dict(year=pac_sc["year"], month=pac_sc["month"], day=1))
pac_sc = pac_sc.sort_values("period").reset_index(drop=True)
pac_sc[["period", "pac_programmed", "pac_executed", "pct_executed_calc", "rank_in_month"]]

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(pac_sc["period"], pac_sc["pac_programmed"], marker="o", label="Programado (PAC actual)")
ax.plot(pac_sc["period"], pac_sc["pac_executed"], marker="o", label="Ejecutado (girado)")
ax.set_title("San Cristóbal — PAC programado vs. ejecutado por mes (unidad por confirmar)")
ax.legend()
ax.grid(alpha=0.3)
plt.show()

In [ ]:
pac_output = PROCESSED_DIR / "f1_pac_fdl_san_cristobal.csv"
pac_sc.drop(columns="period").to_csv(pac_output, index=False)
pac_all.to_csv(PROCESSED_DIR / "f1_pac_fdl_all_localities.csv", index=False)

last = pac_sc.iloc[-1]
register_source("F1", pac_dataset["title"], PAC_DATASET_ID, [r["url"] for r in pac_resources],
                pac_output, f"{int(last['year'])}-{int(last['month']):02d}", len(pac_sc))
print("Guardado:", pac_output)

## 3. F2 — PQRS de la Secretaría Distrital de Gobierno (2022–2026)

Cada fila es una **petición, queja, reclamo o solicitud** radicada. Se filtran las dirigidas a la **Alcaldía Local de San Cristóbal**.

**Decisión de privacidad:** las columnas de texto libre (`OBSERVACION_GENERAL`, `OBSERVACION_SAC`) y las de usuarios internos pueden contener datos personales. **No se conservan.** Para el agente bastan el tema, el tipo, las fechas y el estado.

In [ ]:
PQRS_DATASET_ID = "db9d7291-0308-475e-bd81-cd222c13fca4"
pqrs_dataset = get_dataset(PQRS_DATASET_ID)
pqrs_resource = next(r for r in pqrs_dataset["resources"] if r["format"].upper() == "CSV")
pqrs_path = download(pqrs_resource["url"], "pqrs_sdg.csv")

pqrs_all = pd.read_csv(pqrs_path, sep=";", encoding="utf-8-sig", dtype=str)
print(pqrs_dataset["title"], pqrs_all.shape)
pqrs_all["NOMBRE_LOCALIDAD"].str.strip().value_counts().head(8)

In [ ]:
KEEP = ["RADICADO", "VIGENCIA", "MES_REGISTRO", "RADI_FECH_RADI", "FECHA_RESPUESTA", "MEDIO_RECEPCION",
        "TIPO_PETICION", "TIEMPO_TRAMITE", "ESTADO_ORFEO", "SUBTEMA", "ULTIMA_DEPENDENCIA"]

pqrs = pqrs_all[pqrs_all["NOMBRE_LOCALIDAD"].str.contains("SAN CRISTOBAL", na=False)][KEEP].copy()
pqrs.columns = [normalize_column(c) for c in pqrs.columns]

for column in pqrs.select_dtypes("object"):
    pqrs[column] = pqrs[column].str.strip()

pqrs["filed_at"] = pd.to_datetime(pqrs.pop("radi_fech_radi"), format="%d/%m/%Y %H:%M:%S", errors="coerce")
pqrs["answered_at"] = pd.to_datetime(pqrs.pop("fecha_respuesta"), format="%d/%m/%Y %H:%M:%S", errors="coerce")
pqrs["vigencia"] = pd.to_numeric(pqrs["vigencia"], errors="coerce").astype("Int64")
pqrs["legal_term_days"] = pd.to_numeric(pqrs.pop("tiempo_tramite"), errors="coerce")
pqrs["response_days"] = (pqrs["answered_at"] - pqrs["filed_at"]).dt.days

print("Filas San Cristóbal:", len(pqrs))
print("Valores nulos por columna:")
pqrs.isna().sum()

### Calidad de las fechas
Una respuesta con fecha **anterior** a la radicación es imposible: si aparece, es un error de la fuente y se excluye del cálculo de tiempos (no se borra la fila).

In [ ]:
invalid = pqrs["response_days"] < 0
print(f"Respuestas con fecha anterior a la radicación: {invalid.sum()}")
pqrs.loc[invalid, "response_days"] = pd.NA

pqrs.groupby("vigencia").agg(
    total=("radicado", "count"),
    median_response_days=("response_days", "median"),
    pct_closed=("estado_orfeo", lambda s: (s == "TRAMITE CERRADO").mean().round(3)),
)

> **Nota:** `TIEMPO_TRAMITE` es el plazo legal en **días hábiles** (15 para un derecho de petición), y `response_days` está en **días calendario**. Por eso aquí todavía no se mide el cumplimiento del plazo. Para el MVP hay que calcular días hábiles con el calendario de festivos de Colombia.

In [ ]:
top_topics = pqrs["subtema"].value_counts().head(15)
fig, ax = plt.subplots(figsize=(10, 6))
top_topics.sort_values().plot.barh(ax=ax)
ax.set_title("San Cristóbal — 15 subtemas más frecuentes en PQRS (2022–2026)")
plt.tight_layout()
plt.show()

In [ ]:
pqrs_output = PROCESSED_DIR / "f2_pqrs_san_cristobal.csv"
pqrs.to_csv(pqrs_output, index=False)
register_source("F2", pqrs_dataset["title"], PQRS_DATASET_ID, [pqrs_resource["url"]], pqrs_output,
                str(pqrs["filed_at"].max().date()), len(pqrs))
print("Guardado:", pqrs_output)

## 4. F3 — Presupuestos Participativos

Son las propuestas que **la comunidad votó** para ejecutarse con el presupuesto local. Se usan dos archivos:
- **Avance de propuestas priorizadas:** estado y % de avance de cada propuesta.
- **Presupuesto comprometido por meta:** los contratos/compromisos (CRP) que financian esas propuestas.

**Ojo:** los `.ods` tienen decenas de hojas vacías (restos de enlaces a archivos internos de la entidad). Solo la **primera hoja** tiene datos.

In [ ]:
PP_DATASET_ID = "3ded79df-ec9e-4f92-a949-ca8b1ae12d7d"
pp_dataset = get_dataset(PP_DATASET_ID)

def pick_resource(keyword: str) -> dict:
    return next(r for r in pp_dataset["resources"]
                if keyword in r["url"].lower() and "metadata" not in r["url"].lower())

progress_resource = pick_resource("avance_propuestas_priorizadas")
commitment_resource = pick_resource("presupuesto_comprometido")

def read_first_sheet(resource: dict, filename: str) -> pd.DataFrame:
    frame = pd.read_excel(download(resource["url"], filename), sheet_name=0, engine="odf")
    frame.columns = [normalize_column(c) for c in frame.columns]
    return frame

pp_progress_all = read_first_sheet(progress_resource, "pp_progress.ods")
pp_commitments_all = read_first_sheet(commitment_resource, "pp_commitments.ods")
print("Avance:", pp_progress_all.shape, "| Compromisos:", pp_commitments_all.shape)
print("Columnas de avance:", list(pp_progress_all.columns))

In [ ]:
pp_progress = pp_progress_all[pp_progress_all["no_localidad"] == LOCALITY_CODE].copy()
pp_commitments = pp_commitments_all[pp_commitments_all["no_localidad"] == LOCALITY_CODE].copy()
print(f"Propuestas de San Cristóbal: {len(pp_progress)} | Compromisos: {len(pp_commitments)}")

pd.crosstab(pp_progress["estado_de_la_propuesta"], pp_progress["vigencia_ejecucion"], margins=True)

### Limpieza: sectores escritos de dos formas
La vigencia 2025 escribe los sectores como `Cultura, recreación y deporte` y la 2026 como `CULTURA, RECREACIÓN Y DEPORTE`. Sin normalizar, el mismo sector aparece **dos veces** y los totales quedan partidos. Se unifica en mayúsculas y sin tildes.

In [ ]:
def normalize_label(value) -> str:
    return re.sub(r"\s+", " ", strip_accents(value).upper()).strip()

print("Sectores antes:", pp_progress["sector"].nunique())
for frame in (pp_progress, pp_commitments):
    frame["sector"] = frame["sector"].map(normalize_label)
print("Sectores después:", pp_progress["sector"].nunique())

In [ ]:
progress_by_sector = (pp_progress.groupby("sector")
                      .agg(proposals=("codigo_propuestas", "count"),
                           avg_progress=("pct_de_avance_propuesta", "mean"))
                      .sort_values("proposals", ascending=False))

money_by_sector = (pp_commitments.groupby("sector")["valor_crp_asignado_presupuestos_participativos"]
                   .sum().rename("committed_value"))

progress_by_sector.join(money_by_sector)

In [ ]:
# Data quality: proposals whose financing commitments cannot be found (traceability gap).
financed_codes = set()
for codes in pp_commitments["codigo_propuestas_financiadas_crp"].dropna().astype(str):
    financed_codes.update(re.findall(r"\d+", codes))

pp_progress["has_commitment"] = pp_progress["codigo_propuestas"].astype(str).isin(financed_codes)
print("Propuestas con al menos un compromiso (CRP) asociado:")
pp_progress.groupby("estado_de_la_propuesta")["has_commitment"].mean().round(2)

In [ ]:
pp_progress_output = PROCESSED_DIR / "f3_pp_progress_san_cristobal.csv"
pp_commitments_output = PROCESSED_DIR / "f3_pp_commitments_san_cristobal.csv"
pp_progress.to_csv(pp_progress_output, index=False)
pp_commitments.to_csv(pp_commitments_output, index=False)

cutoff = re.search(r"(\d{2}-\d{2}-\d{4})", progress_resource["url"])
cutoff = cutoff.group(1) if cutoff else "unknown"
register_source("F3-progress", pp_dataset["title"] + " — avance", PP_DATASET_ID, [progress_resource["url"]],
                pp_progress_output, cutoff, len(pp_progress))
register_source("F3-commitments", pp_dataset["title"] + " — compromisos", PP_DATASET_ID,
                [commitment_resource["url"]], pp_commitments_output, cutoff, len(pp_commitments))
print("Guardado:", pp_progress_output, pp_commitments_output)

## 5. Trazabilidad de las fuentes

El PRD exige que **toda respuesta del agente cite su fuente con fecha de corte**. Este archivo (`sources_metadata.json`) es el inicio de esa trazabilidad: dice de dónde salió cada archivo procesado y con qué fecha de corte.

In [ ]:
metadata_path = PROCESSED_DIR / "sources_metadata.json"
metadata_path.write_text(json.dumps(SOURCES_METADATA, ensure_ascii=False, indent=2), encoding="utf-8")
pd.DataFrame(SOURCES_METADATA)[["source_id", "title", "data_cutoff", "rows", "output_file"]]

In [ ]:
# Optional (Colab only): download all processed files as a zip.
try:
    from google.colab import files
    import shutil
    shutil.make_archive("processed_san_cristobal", "zip", PROCESSED_DIR)
    files.download("processed_san_cristobal.zip")
except ImportError:
    print("No se está ejecutando en Colab: los archivos quedan en", PROCESSED_DIR.resolve())

## 6. Hallazgos y tareas para el equipo

### Hallazgos de esta exploración
1. **El POAI no está en el portal de Datos Abiertos.** Hay que conseguirlo en el sitio de la Alcaldía Local (fuente F5 del PRD).
2. **F1 no declara la unidad** de los valores de PAC. Parecen millones de pesos, pero no está confirmado.
3. **F1 publica un CSV por mes con formato de reporte**, no de tabla. Esa fragilidad es en sí misma un hallazgo para la Alcaldía: el formato dificulta la reutilización.
4. **F1 tiene valores vacíos**: por ejemplo, Kennedy en diciembre de 2024 reporta 105 % de ejecución, pero el valor ejecutado está en blanco. No se inventa: se deja vacío y el agente debe decir que el dato no está.
5. **F1 está desactualizado**: el último mes publicado es junio de 2025, mientras que F2 y F3 llegan hasta 2026. Comparar fuentes con cortes distintos exige mostrar siempre la fecha de corte.
6. **F2 contiene texto libre con posibles datos personales**: se descartó por privacidad.
7. **F3 trae decenas de hojas vacías** con rutas internas de la entidad, y los sectores están escritos de dos formas distintas entre vigencias: más fricción para cualquier ciudadano que quiera usar el dato.

Los puntos 3, 4, 5 y 7 son justamente la brecha que plantea el reto: *el dato existe, pero no es accesible ni utilizable*.

### Tareas (escribe tu nombre al lado de la que tomes)
- [ ] Confirmar la **unidad de F1** (¿millones de pesos?) con la Secretaría de Hacienda o en documentación oficial. — *Responsable:*
- [ ] Conseguir el **POAI vigente** de San Cristóbal (PDF/XLSX) y subirlo a la carpeta del proyecto. — *Responsable:*
- [ ] Revisar los **15 subtemas de PQRS** y proponer la **taxonomía de temas** para clasificar las preguntas y los vacíos (seguridad, vías, salud…). — *Responsable:*
- [ ] Escribir **20 preguntas ciudadanas reales** que estos datos SÍ pueden responder, con su respuesta esperada (inicio del set de evaluación del PRD). — *Responsable:*
- [ ] Escribir **10 preguntas** que estos datos NO pueden responder y qué dato haría falta (vacíos de información). — *Responsable:*

### Observaciones del equipo
*(Escribe aquí lo que notes en los datos: cosas raras, preguntas, ideas.)*